# FLAME checkpoint parity on T4
Upload `flame-source.zip` built from the local ENNX checkout (see `docs/flame.md`). No model token is needed. This downloads approximately 2.6 GB of weights, then compares FP32 JAX evaluation with independent PyTorch equations. This is not released-Megatron parity or a BO benchmark.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
import subprocess
import sys
import os
os.chdir('/content')
if not Path('flame-source.zip').exists():
    uploaded = files.upload()
    assert 'flame-source.zip' in uploaded
with zipfile.ZipFile('flame-source.zip') as archive:
    archive.extractall('/content/flame-source')
os.chdir('/content/flame-source')
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
assert 'T4' in gpu, f'Expected a T4 runtime, got {gpu}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'ops/flame/requirements-cuda.txt'], check=True)
subprocess.run([sys.executable, '-c', "import jax; d = jax.devices()[0]; assert d.platform == 'gpu' and 'T4' in d.device_kind, d; print(d)"], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests/testflame.py', '-q'], check=True)
checkpoint = Path('/content/flame-checkpoint')
if not (checkpoint / 'manifest.json').exists():
    subprocess.run([sys.executable, '-m', 'ops.flame.checkpoint', '--output', str(checkpoint)], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'ops.flame.parity', str(checkpoint), '--tokens', 'ops/flame/tokens.json', '--output', '/content/flame-parity.json'], check=True)
Path('/content/flame-environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
files.download('/content/flame-parity.json')
files.download('/content/flame-environment.txt')

In [ ]:
import treescope
from ops.flame import model
config, params = model.load_checkpoint(checkpoint)
treescope.display(model.parameter_tree(params))